# Data Cleaning Checklist for Event Dataset


Dataset columns: 
- `user_id`,
-  `session_id`, 
-  `event`, 
-  `page`, 
-  `category`, 
-  `timestamp`, 
-  `device`, 
-  `country`, 
-  `user_type`


In [ ]:
#--------------
# Import libreries
# -------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")


In [326]:
#--------------
# Load raw dataset
# -------------

raw_path = os.path.join("..", "data", "raw", "ecommerce_dataset_raw_realistic.csv")
df = pd.read_csv(raw_path)

print("Database shape:", df.shape)
df.head(5)

Database shape: (2944, 9)


,user_id,session_id,event,page,category,timestamp,device,country,user_type
0,103,s103_2,product_view,product_page,Sports,2026-03-03 06:02:00,mobile,IT,returning
1,221,s221_1,product_view,product_page,ACCESSORIES,2026-03-09 21:15:19,desktop,IT,new
2,291,s291_2,page_view,homepage,NaN,2026-03-05 05:02:52,NaN,ES,new
3,278,s278_1,page_view,homepage,NaN,2026-03-03 14:39:06,desktop,ES,new
4,492,s492_4,page_view,homepage,NaN,2026-03-07 20:33:17,desktop,FR,new


In [327]:
# ==========================
# QUICK EDA OVERVIEW
# ==========================

print("=== DATASET SHAPE ===")
print("Rows", df.shape[0])
print("Columns:", df.shape[1])

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== BASIC COUNTS ===")
print("Total events:", len(df))
print("Unique users:", df['user_id'].nunique())
print("Unique sessions:", df['session_id'].nunique())

print("\n=== UNIQUE VALUES IN CATEGORICAL COLUMNS ===")
cat_cols = ['event', 'page', 'category', 'device', 'country', 'user_type']
print(df[cat_cols].nunique().sort_values())

print("\n=== EVENT DISTRIBUTION ===")
print(df['event'].value_counts())

print("\n=== TIME RANGE ===")
print("Min timestamp:", df['timestamp'].min())
print("Max timestamp:", df['timestamp'].max())

=== DATASET SHAPE ===
Rows 2944
Columns: 9

=== DATA TYPES ===
user_id       int64
session_id      str
event           str
page            str
category        str
timestamp       str
device          str
country         str
user_type       str
dtype: object

=== MISSING VALUES ===
user_id         0
session_id      0
event           0
page            0
category      971
timestamp       0
device        257
country       128
user_type       0
dtype: int64

=== BASIC COUNTS ===
Total events: 2944
Unique users: 500
Unique sessions: 1477

=== UNIQUE VALUES IN CATEGORICAL COLUMNS ===
device       2
user_type    2
country      4
page         4
event        5
category     8
dtype: int64

=== EVENT DISTRIBUTION ===
event
page_view       1515
product_view    1082
add_to_cart      281
checkout          58
purchase           8
Name: count, dtype: int64

=== TIME RANGE ===
Min timestamp: 2026-03-01 08:08:18
Max timestamp: 2026-03-10 21:59:36


## Data Cleaning

- **1. Review columns and data types**
  
| Column     | Tipo actual | Observación              |
| ---------- | ----------- | ------------------------ |
| user_id    | int64       | OK                       |
| session_id | str         | OK                       |
| event      | str         | OK                       |
| page       | str         | OK                       |
| category   | str         | **971 missing**          |
| timestamp  | str         | **convert to datetime** |
| device     | str         | **257 missing**          |
| country    | str         | **128 missing**          |
| user_type  | str         | OK                       |


In [328]:
# Convert timestamp to datetime
# First of all I make a copy for cleaning/processing
df_clean = df.copy()

df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])

In [329]:
# Verify timestamp datatype
df_clean.dtypes

user_id                int64
session_id               str
event                    str
page                     str
category                 str
timestamp     datetime64[us]
device                   str
country                  str
user_type                str
dtype: object

- **2. Missing values**
 
 - category: 971 missing → ~33% of data, important for events
 - device: 257 missing
 - country: 128 missing 

In [330]:
# Fill missing categorical values
df_clean['category'] = df_clean['category'].fillna("unknown")
df_clean['device'] = df_clean['device'].fillna("unknown")
df_clean['country'] = df_clean['country'].fillna("unknown")

- **3.Parsing text**
  
  - Detect blank spaces
  - Detect uppercase / lowercase
  - Normalize text (strip + lowercase)

In [331]:
# Find values with leading/trailing whitespace
cat_cols = ['event','page','category','device','country','user_type']

for col in cat_cols:
    mask = df_clean[col].str.startswith(' ') | df_clean[col].str.endswith(' ')

    if mask.sum() > 0:
        print(f"\nColumn with whitespace issue: {col}")
        print(df_clean.loc[mask, col].unique())

In [332]:
# Detect variants in each column
cat_cols = ['event','page','category','device','country','user_type']

for col in cat_cols:
    
    original_vals = sorted(df_clean[col].dropna().unique())
    lower_vals = sorted(df_clean[col].dropna().str.lower().unique())
    
    print(f"\n====== {col.upper()} ======")
    print("Original values:")
    print(original_vals)
    
    print("\nLowercase normalized values:")
    print(lower_vals)
    
    if len(original_vals) != len(lower_vals):
        print("\n⚠️ Possible case inconsistencies detected")


====== EVENT ======
Original values:
['add_to_cart', 'checkout', 'page_view', 'product_view', 'purchase']

Lowercase normalized values:
['add_to_cart', 'checkout', 'page_view', 'product_view', 'purchase']

====== PAGE ======
Original values:
['checkout', 'confirmation', 'homepage', 'product_page']

Lowercase normalized values:
['checkout', 'confirmation', 'homepage', 'product_page']

====== CATEGORY ======
Original values:
['ACCESSORIES', 'Accessories', 'Clothing', 'Electronics', 'Home', 'Sports', 'clothing', 'homepage', 'unknown']

Lowercase normalized values:
['accessories', 'clothing', 'electronics', 'home', 'homepage', 'sports', 'unknown']

⚠️ Possible case inconsistencies detected

====== DEVICE ======
Original values:
['desktop', 'mobile', 'unknown']

Lowercase normalized values:
['desktop', 'mobile', 'unknown']

====== COUNTRY ======
Original values:
['DE', 'ES', 'FR', 'IT', 'unknown']

Lowercase normalized values:
['de', 'es', 'fr', 'it', 'unknown']

====== USER_TYPE ======
Or

In [333]:
# Normalize text (strp + lower)
cat_cols = ['event','page','category','device','country','user_type']

for col in cat_cols:
    df_clean[col] = df_clean[col].str.strip().str.lower()

In [334]:
# Check number of unique categories pos-normalization 
# 'unknown' add one category 
# ACCESSORIES and Accessories are one category, then there are 7 categories in category
df_clean[cat_cols].nunique()

event        5
page         4
category     7
device       3
country      5
user_type    2
dtype: int64

In [335]:
# Number of unique in raw dataset
df[cat_cols].nunique()

event        5
page         4
category     8
device       2
country      4
user_type    2
dtype: int64

- 4. **Detect and remove duplicates**

In [336]:
# Detect duplicates in df_clean
dup_rows = df_clean.duplicated(keep=False)  # show all duplicated rows
total_duplicates = dup_rows.sum()           # use the same variable
print(f"Total duplicate rows: {total_duplicates}")

# Show all duplicated rows, ordered by columns
if total_duplicates > 0:
    duplicate_rows = df_clean[dup_rows].sort_values(by=df_clean.columns.tolist())
    print("\nDuplicated rows:")
    print(duplicate_rows)

Total duplicate rows: 116

Duplicated rows:
      user_id session_id         event          page  category  \
52         24     s024_1  product_view  product_page      home   
1223       24     s024_1  product_view  product_page      home   
266        44     s044_1  product_view  product_page  clothing   
1462       44     s044_1  product_view  product_page  clothing   
220        54     s054_2  product_view  product_page    sports   
...       ...        ...           ...           ...       ...   
2624      475     s475_3  product_view  product_page   unknown   
1034      482     s482_4     page_view      homepage  homepage   
2154      482     s482_4     page_view      homepage  homepage   
197       491     s491_3  product_view  product_page  clothing   
1724      491     s491_3  product_view  product_page  clothing   

               timestamp   device  country  user_type  
52   2026-03-08 22:47:40   mobile  unknown        new  
1223 2026-03-08 22:47:40   mobile  unknown        n

There are 58 exact duplicated rows

In [ ]:
# Drop exact duplicates
df_clean = df_clean.drop_duplicates()

# Verify there are no duplicates
dup_rows_after = df_clean.duplicated(keep=False)
print("Remaining duplicate rows:", dup_rows_after.sum())

Remaining duplicate rows: 0


Since the dataset simulates tracking data from **Google Analytics**, duplicated events were identified based on the combination of `user_id`, `session_id`, `event`, and `timestamp`.
These fields uniquely define an event in the analytics pipeline.
Duplicate rows were removed to avoid overcounting events.

In [ ]:
# Detect duplicates based on all event-defining fields
dup_mask = df_clean.duplicated(subset=['user_id','session_id','event','timestamp'], keep=False)
total_dupes = dup_mask.sum()
print(f"Total duplicate event rows: {total_dupes}")

# Show duplicates
if total_dupes > 0:
    dup_rows = df_clean[dup_mask].sort_values(by=['user_id','session_id','timestamp','event'])
    print("\nDuplicated event rows:")
    print(dup_rows)

Total duplicate event rows: 0


In [ ]:
# Drop duplicates based on event: 'user_id','session_id','event','timestamp'
df_clean = df_clean.drop_duplicates(subset=['user_id','session_id','event','timestamp'], keep='first')

# Verificar
remaining_dupes = df_clean.duplicated(subset=['user_id','session_id','event','timestamp'], keep=False).sum()
print("Remaining duplicate event rows:", remaining_dupes)

Remaining duplicate event rows: 0


- 5. **Convert `event`,`page`,`category`,`device`, `country`,`user_type` to category datatype**

In [ ]:
cat_cols = ['event','page','category','device','country','user_type']

for col in cat_cols:
    df_clean[col] = df_clean[col].astype('category')     

In [ ]:
Final verification
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2944 entries, 0 to 2943
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     2944 non-null   int64         
 1   session_id  2944 non-null   str           
 2   event       2944 non-null   category      
 3   page        2944 non-null   category      
 4   category    2944 non-null   category      
 5   timestamp   2944 non-null   datetime64[us]
 6   device      2944 non-null   category      
 7   country     2944 non-null   category      
 8   user_type   2944 non-null   category      
dtypes: category(6), datetime64[us](1), int64(1), str(1)
memory usage: 86.6 KB


In [ ]:
print("\n=== sample of cleaner data ===")
df_clean.head()


=== sample of cleaner data ===


,user_id,session_id,event,page,category,timestamp,device,country,user_type
0,103,s103_2,product_view,product_page,sports,2026-03-03 06:02:00,mobile,it,returning
1,221,s221_1,product_view,product_page,accessories,2026-03-09 21:15:19,desktop,it,new
2,291,s291_2,page_view,homepage,unknown,2026-03-05 05:02:52,unknown,es,new
3,278,s278_1,page_view,homepage,unknown,2026-03-03 14:39:06,desktop,es,new
4,492,s492_4,page_view,homepage,unknown,2026-03-07 20:33:17,desktop,fr,new


In [338]:
df_clean.to_csv("df_clean.csv", index=False)